In [1]:
import numpy as np
from scipy.sparse import random as sparse_random, csr_matrix
import matplotlib.pyplot as plt
import numba as nb
from scipy.sparse.linalg import eigsh
import time

# Matrix Generation
def generate_sparse_matrix(size, density):
    diagonal = np.linspace(-1, 1, size)
    rvs = np.random.randn  
    upper = sparse_random(size, size, density=density/2, data_rvs=rvs, format='coo')
    A = upper + upper.T
    A.setdiag(diagonal)
    return A.tocsr()

# Block CSR Matrix-Vector Multiplication (Numba Accelerated)
@nb.njit(parallel=True, fastmath=True, cache=True)
def block_csr_mv(A_data, A_indices, A_indptr, X):
    n = A_indptr.shape[0] - 1
    b = X.shape[1]
    Y = np.zeros((n, b), dtype=X.dtype)
    for i in nb.prange(n):
        for j in range(A_indptr[i], A_indptr[i+1]):
            col = A_indices[j]
            for k in range(b):
                Y[i, k] += A_data[j] * X[col, k]
    return Y

def _gamma_from_sorted(vals, clip=1e9):
    if vals.shape[0] < 3:
        return None
    lam1, lam2, lam_max = vals[0], vals[1], vals[-1]
    denom = lam2 - lam1
    if denom <= 0:
        return None
    g = (lam_max - lam2) / denom
    if not np.isfinite(g):
        return None
    return min(g, clip)

def _true_gamma_eigsh(A, clip=1e9):
    try:
        lam_small = np.sort(eigsh(A, k=2, which='SA', return_eigenvectors=False))
        lam1, lam2 = lam_small[0], lam_small[1]
        lam_max = eigsh(A, k=1, which='LA', return_eigenvectors=False)[0]
        denom = lam2 - lam1
        if denom <= 0:
            return None
        g = (lam_max - lam2) / denom
        if not np.isfinite(g):
            return None
        return min(g, clip)
    except Exception:
        return None

@nb.njit(fastmath=True, cache=True)
def block_lanczos_loop(A_data, A_indices, A_indptr, Q, T_blocks, m, b, ortho_thresh):
    n = Q.shape[0]
    eps = np.finfo(np.float64).eps
    m_eff = m  # effective number of block iterations
    for k in range(m):
        # Indices of block k.
        start = k * b
        end = (k+1) * b
        Qk = Q[:, start:end]

        # Compute new block: V = A * Qk.
        V = block_csr_mv(A_data, A_indices, A_indptr, Qk)

        # Compute block alpha = Qk^T V, (b,b) matrix.
        Qk_contig = np.ascontiguousarray(Qk)
        V_contig = np.ascontiguousarray(V)
        alpha = np.dot(Qk_contig.T, V_contig)

        # Store alpha in diagonal block T_blocks[k,k].
        for i in range(b):
            for j in range(b):
                T_blocks[k, k, i, j] = alpha[i, j]

        # V <- V - Qk * alpha
        V = V - np.dot(Qk_contig, np.ascontiguousarray(alpha))

        # Subtract previous block contribution if k > 0: V <- V - Q_{k-1} * beta_{k,k-1}
        if k > 0:
            Q_prev = Q[:, (k-1)*b : k*b]
            beta_prev = T_blocks[k, k-1]  # (b,b)
            V = V - np.dot(np.ascontiguousarray(Q_prev), np.ascontiguousarray(beta_prev))

        # Full reorthogonalization (two passes, dynamic threshold)
        norm_V = np.sqrt(np.sum(V * V))
        dyn_thresh = max(ortho_thresh, np.sqrt(eps) * norm_V)
        for rep in range(2):
            for j in range(k+1):
                Qj = Q[:, j*b:(j+1)*b]
                P = np.dot(np.ascontiguousarray(Qj).T, np.ascontiguousarray(V))
                proj_norm = np.sqrt(np.sum(P * P))  # Frobenius
                if proj_norm > dyn_thresh:
                    V = V - np.dot(np.ascontiguousarray(Qj), P)

        # QR(V) -> next block and R
        Q_new, R = np.linalg.qr(V)

        # Store Q_new into next block of Q.
        next_start = (k+1)*b
        next_end = (k+2)*b
        Q[:, next_start:next_end] = Q_new

        # Store ONLY the lower off-diagonal block beta = R
        T_blocks[k+1, k] = R
        # DO NOT write T_blocks[k, k+1] = R.T (out-of-bounds when k == m-1)

        # Breakdown test
        rnorm = np.sqrt(np.sum(R * R))
        if rnorm == 0:
            m_eff = k + 1
            return m_eff

    return m_eff

# Implicit Restarted Lanczos (IRL) with Deflation and Shift Selection
def block_irl(matrix_s, v_init, tol=1e-8, m=None, max_iter=40000, max_m=100, ortho_thresh=1e-10, k=1, record_gamma=True, compute_true_gamma=False,
              gamma_clip=1e6):
    n = matrix_s.shape[0]
    if m is None:
        m = max(2*k+1, 40)

    Q = np.zeros((n, (m+1)*k), dtype=np.float64)
    T_blocks = np.zeros((m+1, m, k, k), dtype=np.float64)

    # Ensure starting block shape and orthonormality
    if v_init.ndim == 1:
        v_init = v_init.reshape(-1, 1)
        if k > 1:
            v_init = np.repeat(v_init, k, axis=1)
    # Orthonormalize the initial block (good even if k==1)
    Q0, _ = np.linalg.qr(v_init)
    Q[:, :k] = Q0[:, :k]

    A = matrix_s.tocsr()
    A_data, A_indices, A_indptr = A.data, A.indices, A.indptr

    total_iters = 0
    instantaneous_sizes = []
    last_ritz_res = np.inf

    gamma_true = None
    gamma_estimates = []
    if compute_true_gamma:
        gamma_true = _true_gamma_eigsh(A, clip=gamma_clip)

    while total_iters < max_iter:
        m_eff = block_lanczos_loop(A_data, A_indices, A_indptr, Q, T_blocks, m, k, ortho_thresh)
        total_iters += m_eff
        instantaneous_sizes.append(m_eff * k)

        # Assemble dense block-tridiagonal T
        T_dense = np.zeros((m_eff*k, m_eff*k), dtype=np.float64)
        for i in range(m_eff):
            T_dense[i*k:(i+1)*k, i*k:(i+1)*k] = T_blocks[i, i]
            if i < m_eff - 1:
                B = T_blocks[i+1, i]
                T_dense[(i+1)*k:(i+2)*k, i*k:(i+1)*k] = B
                T_dense[i*k:(i+1)*k, (i+1)*k:(i+2)*k] = B.T

        eigvals, eigvecs = np.linalg.eigh(T_dense)  # ascending

        # Per-restart gamma estimate (from Ritz values)
        if record_gamma:
            g_est = _gamma_from_sorted(eigvals, clip=gamma_clip)
            gamma_estimates.append(g_est)

        # --- Residual estimate using the TRUE tail block B_m ---
        if m_eff < 1:
            ritz_res = np.inf
        else:
            Bm = T_blocks[m_eff, m_eff-1]  # (k,k)
            if k == 1:
                y_last = eigvecs[-1, 0]   # last component of the target Ritz vector
                beta = Bm[0, 0]
                ritz_res = abs(beta * y_last)
            else:
                # take the max residual over the first k Ritz vectors (smallest)
                ritz_res = 0.0
                for j in range(k):
                    y_tail = eigvecs[-k:, j]
                    rj = np.linalg.norm(Bm @ y_tail)
                    if rj > ritz_res:
                        ritz_res = rj

        # Convergence (relative to the smallest Ritz value)
        lam_target = eigvals[0]
        if ritz_res < tol * max(1.0, abs(lam_target)):
            approx_eigvecs = Q[:, :m_eff*k] @ eigvecs[:, :k]
            return total_iters, eigvals[:k], approx_eigvecs, instantaneous_sizes, gamma_true, gamma_estimates

        # --- implicit restarting (shifts = all unwanted) ---
        target_ind = 0
        unwanted = np.delete(eigvals, np.arange(target_ind, target_ind+k))
        shifts = np.sort(unwanted)[::-1] if unwanted.size else np.array([])

        V_plus = np.eye(m_eff*k)
        T_temp = T_dense.copy()
        for mu in shifts:
            Qs, Rs = np.linalg.qr(T_temp - mu * np.eye(m_eff*k))
            T_temp = Rs @ Qs + mu * np.eye(m_eff*k)
            V_plus = V_plus @ Qs

        # New starting block
        new_block = Q[:, :m_eff*k] @ V_plus[:, :k]
        Q_new, _ = np.linalg.qr(new_block)
        Q[:, :k] = Q_new

        # Mild adaptive growth of m if stalled
        if ritz_res > 0.8 * last_ritz_res and m < max_m:
            new_m = min(int(m * 1.15), max_m)
            if new_m > m:
                m = new_m
                new_Q = np.zeros((n, (m+1)*k), dtype=np.float64)
                new_Q[:, :k] = Q[:, :k]
                Q = new_Q
                T_blocks = np.zeros((m+1, m, k, k), dtype=np.float64)

        last_ritz_res = ritz_res

    # Fallback (max_iter hit)
    approx_eigvecs = Q[:, :m_eff*k] @ eigvecs[:, :k]
    return total_iters, eigvals[:k], approx_eigvecs, instantaneous_sizes, gamma_true, gamma_estimates

# ---- quick warmup so Numba compiles upfront ----
def _warmup_numba():
    # tiny 16x16 toy to JIT both kernels
    n, b = 16, 1
    from scipy.sparse import diags
    A = diags([np.ones(n-1), 2*np.ones(n), np.ones(n-1)], [-1,0,1]).tocsr()
    A_data, A_indices, A_indptr = A.data, A.indices, A.indptr
    X = np.random.randn(n, b)
    _ = block_csr_mv(A_data, A_indices, A_indptr, X)  # JIT 1
    # block_lanczos_loop warmup
    m = 4
    Q = np.zeros((n, (m+1)*b))
    Q[:, :b] = np.linalg.qr(np.random.randn(n, b))[0]
    T_blocks = np.zeros((m+1, m, b, b))
    _ = block_lanczos_loop(A_data, A_indices, A_indptr, Q, T_blocks, m, b, 1e-12)

# ---- faster true-gamma using eigsh with relaxed tol/maxiter ----
def _true_gamma_eigsh_fast(A, clip=1e9, tol=1e-8, maxiter=10000):
    try:
        lam_small = np.sort(eigsh(A, k=2, which='SA', return_eigenvectors=False,
                                  tol=tol, maxiter=maxiter))
        lam1, lam2 = lam_small[0], lam_small[1]
        lam_max = eigsh(A, k=1, which='LA', return_eigenvectors=False,
                        tol=tol, maxiter=maxiter)[0]
        denom = lam2 - lam1
        if denom <= 0:
            return None
        g = (lam_max - lam2) / denom
        if not np.isfinite(g):
            return None
        return min(g, clip)
    except Exception:
        return None

# ---- evaluation that prints per-density summary immediately ----
def evaluate_gamma_true_and_irl(size, density_levels, num_trials=40, k=1, tol_irl=1e-8, max_iter_irl=10000, ortho_thresh=1e-10,
                                gamma_tol=1e-8, gamma_maxiter=10000,print_live=True):
    _warmup_numba()  # compile numba kernels once
    avg_table = {}

    for density in density_levels:
        print(f"\n--- Evaluating Density Level: {density*100:.2f}% ---", flush=True)
        g_true_list = []
        g_irl_list  = []
        basis_list  = []

        for _ in range(num_trials):
            A  = generate_sparse_matrix(size, density)
            v0 = np.random.rand(size, k)

            # --- true gamma from the ORIGINAL matrix (λ1, λ2, λmax) ---
            g_true = _true_gamma_eigsh_fast(A, clip=1e6, tol=gamma_tol, maxiter=gamma_maxiter)

            # --- run IRL once to get basis + IRL γ from final T_m ---
            total_iters, _, _, inst_sizes, _, gamma_estimates = block_irl(
                A, v0, tol=tol_irl, m=None, max_iter=max_iter_irl, max_m=100,
                ortho_thresh=ortho_thresh, k=k,
                record_gamma=True, compute_true_gamma=False
            )
            basis = float(np.sum(inst_sizes))

            # last valid IRL gamma (from Ritz spectrum)
            g_irl = None
            for ge in reversed(gamma_estimates):
                if ge is not None and np.isfinite(ge):
                    g_irl = float(ge)
                    break

            if g_true is not None and np.isfinite(g_true):
                g_true_list.append(float(g_true))
            if g_irl is not None:
                g_irl_list.append(float(g_irl))
            basis_list.append(basis)

        # averages (ignore missing gammas)
        avg_gamma_true = float(np.mean(g_true_list)) if len(g_true_list) else float('nan')
        avg_gamma_irl  = float(np.mean(g_irl_list))  if len(g_irl_list)  else float('nan')
        avg_basis      = float(np.mean(basis_list))  if len(basis_list)  else float('nan')

        avg_table[density] = {
            'avg_gamma_true': avg_gamma_true,
            'avg_gamma_irl':  avg_gamma_irl,
            'avg_basis':      avg_basis,
            'n_used_true':    len(g_true_list),
            'n_used_irl':     len(g_irl_list),
            'n_trials':       num_trials
        }

        if print_live:
            dens_pct = 100.0 * density
            gT = f"{avg_gamma_true:.6g}" if np.isfinite(avg_gamma_true) else "nan"
            gI = f"{avg_gamma_irl:.6g}"  if np.isfinite(avg_gamma_irl)  else "nan"
            bA = f"{avg_basis:.6g}"      if np.isfinite(avg_basis)      else "nan"
            print(f"Density {dens_pct:6.2f}%:  avg γ(true) = {gT:>2} , avg γ(IRL) = {gI:>2}  ,  avg basis = {bA:>2}  ")

    return avg_table

size = 10000
density_levels = [0.01, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45,
                  0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 1.0]
k = 1

avg_table_both = evaluate_gamma_true_and_irl(
    size, density_levels,
    num_trials=40, k=k,
    tol_irl=1e-8, max_iter_irl=10000,
    ortho_thresh=1e-10,
    gamma_tol=1e-8, gamma_maxiter=10000,
    print_live=True)


--- Evaluating Density Level: 1.00% ---
Density   1.00%:  avg γ(true) = 1630.16 , avg γ(IRL) = 832.778  ,  avg basis = 402.35  

--- Evaluating Density Level: 5.00% ---
Density   5.00%:  avg γ(true) = 1174.62 , avg γ(IRL) = 737.818  ,  avg basis = 362  

--- Evaluating Density Level: 10.00% ---
Density  10.00%:  avg γ(true) = 1547.24 , avg γ(IRL) = 785.079  ,  avg basis = 402.35  

--- Evaluating Density Level: 15.00% ---
Density  15.00%:  avg γ(true) = 1562.12 , avg γ(IRL) = 972.375  ,  avg basis = 385.275  

--- Evaluating Density Level: 20.00% ---
Density  20.00%:  avg γ(true) = 1132.02 , avg γ(IRL) = 754.918  ,  avg basis = 347.05  

--- Evaluating Density Level: 25.00% ---
Density  25.00%:  avg γ(true) = 1229.38 , avg γ(IRL) = 866.074  ,  avg basis = 360.55  

--- Evaluating Density Level: 30.00% ---
Density  30.00%:  avg γ(true) = 1277.71 , avg γ(IRL) = 849.753  ,  avg basis = 367.25  

--- Evaluating Density Level: 35.00% ---
Density  35.00%:  avg γ(true) = 1351.61 , avg γ(IRL)